# SQL SPA Tools 自動化測試框架

使用合成資料驗證工具頁、個資辨識及 Oracle SQL Formatter 批次處理效能，並產生 Excel 報告與簡報圖表。全程不連線 Oracle，也不使用正式資料。

操作細節請參閱 `colab/docs/testing-guide.md`。

In [ ]:
# 執行設定：quick 產生 50 案例；full 產生完整 1,000 案例。
MODE = 'full'
SEED = 20261023
SOURCE_URL = ''  # 可留空並於下一步上傳專案 ZIP；若填 Git URL，執行時會 clone 該來源。
SKIP_BROWSER = False
SKIP_PERFORMANCE = False

In [ ]:
# 專案 ZIP 上傳後，下一個設定儲存格會依該版本的 requirements.txt 安裝套件。
print('請繼續執行下一格上傳或取得專案。')

> 請上傳 ZIP，或在設定格提供可讀取的 Git URL。套件會由專案內的 `colab/requirements.txt` 安裝。

In [ ]:
from pathlib import Path
import shutil, subprocess, sys, zipfile
from google.colab import files

WORK = Path('/content/sql-spa-project')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
if SOURCE_URL.strip():
    subprocess.run(['git', 'clone', '--depth', '1', SOURCE_URL, str(WORK / 'repo')], check=True)
    candidates = [WORK / 'repo']
else:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1: raise ValueError('請上傳一個包含 sql-spa-tools-main/ 與 colab/ 的專案 ZIP。')
    with zipfile.ZipFile(zip_names[0]) as archive: archive.extractall(WORK / 'unpacked')
    candidates = [p.parent for p in (WORK / 'unpacked').rglob('sql-spa-tools-main') if p.is_dir()]
project = next((p for p in candidates if (p / 'sql-spa-tools-main').is_dir()), None)
if project is None: raise FileNotFoundError('ZIP 內找不到 sql-spa-tools-main/。請上傳專案根目錄 ZIP，不要只上傳 sql-spa-tools-main 資料夾。')
runner = project / 'colab' / 'testing' / 'runner.py'
if not runner.is_file(): raise FileNotFoundError(f'找不到測試執行器：{runner}。ZIP 必須同時包含 sql-spa-tools-main/ 與 colab/；請重新下載完整專案 ZIP。')
req = project / 'colab' / 'requirements.txt'
if req.is_file(): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=True)
else: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib>=3.8,<4', 'xlsxwriter>=3.2,<4', 'playwright>=1.50,<2'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True, stdout=subprocess.DEVNULL)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-noto-cjk'], check=True, stdout=subprocess.DEVNULL)
subprocess.run(['fc-cache', '-f'], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
font_probe = subprocess.run(['fc-list', ':lang=zh', 'family'], check=True, capture_output=True, text=True)
if not font_probe.stdout.strip(): raise RuntimeError('系統找不到中文字型。請確認 fonts-noto-cjk 安裝成功後重新執行此儲存格。')
print('可用中文字型：', ', '.join(sorted(set(font_probe.stdout.splitlines()))[:5]))
# 清除舊的 Matplotlib 字型索引，確保後續測試程序讀到剛安裝的 CJK 字型。
import matplotlib
font_cache = Path(matplotlib.get_cachedir())
for cache_file in font_cache.glob('fontlist-v*.json'): cache_file.unlink()
subprocess.run(['npm', 'install', '--prefix', '/content/sqlspa-node', '--no-save', 'node@22'], check=True)
import os
os.environ['SQLSPA_NODE_BIN'] = '/content/sqlspa-node/node_modules/node/bin/node'
# Chromium 依賴 ATK 等 Linux 系統函式庫；Colab 需一併安裝瀏覽器系統相依套件。
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True)
print('專案路徑：', project)

In [ ]:
import json
cmd = [sys.executable, str(runner), '--mode', MODE, '--seed', str(SEED), '--project-dir', str(project)]
if SKIP_BROWSER: cmd.append('--skip-browser')
if SKIP_PERFORMANCE: cmd.append('--skip-performance')
run = subprocess.run(cmd, cwd=project, text=True, capture_output=True)
print(run.stdout)
if run.stderr: print(run.stderr[-6000:])
output_line = next((line for line in run.stdout.splitlines() if line.startswith('{')), None)
if output_line is None: raise RuntimeError('測試執行器沒有回傳成果路徑。')
summary = json.loads(output_line)
result_zip = Path(summary['output_dir']) / 'test-results.zip'
print('測試結束，狀態碼：', run.returncode, '；報告包：', result_zip)

In [ ]:
# 即使有測試失敗仍提供完整報告供檢視。
files.download(str(result_zip))
if run.returncode: print('有失敗或執行錯誤；請查看 Excel 的「失敗與限制」及 results/results.json。')